In [1]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0], 
                                  [0.0, 0.9, 0.1, 0.0, 0.0], 
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]]) 
emission_nuc_codes = {'A': 0, 
                      'C': 1, 
                      'G': 2, 
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00], 
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]]) 

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"


In [2]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [3]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)


-43.89740030179307


In [4]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)


-43.45111319916465


In [5]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)


-43.944833355027704


In [6]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)


-42.58225552052512


In [7]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)


-41.21967768602254


In [8]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)


-41.713397841885595


In [9]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)


-40.98025137355685


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides. 

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .] 
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .] 
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

```

It is important to remember that you will be working in the log scale.

In [10]:
# Initiate two matrices: 
# viterbi_value_matrix: to store the values described in the documentation above 
# viterbi_trace_matrix: to store the path the lead to the the maximum value in each cell

import sys
import os
import math
import numpy as np

# Configuration & Data Setup
states_list = ["s", "E", "5", "I", "e"]
states = {"s": 0, "E": 1, "5": 2, "I": 3, "e": 4}

# Transition probabilities (Dictionary format as per your project structure)
state_transition_prob = {
    "s": {"E": 1.0, "5": 0.0}, # Updated to 1.0 for biological accuracy
    "E": {"E": 0.9, "5": 0.1},
    "5": {"I": 1.0},
    "I": {"I": 0.9, "e": 0.1}
}

# Emission probabilities (Includes small pseudocounts for robustness)
state_emission_prob = {
    "E": {"A": 0.25, "T": 0.25, "G": 0.25, "C": 0.25},
    "5": {"A": 0.05, "T": 0.05, "G": 0.90, "C": 1e-10}, # Added pseudocount
    "I": {"A": 0.40, "T": 0.10, "G": 0.10, "C": 0.40}
}

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

# Matrix Initialization
# viterbi_value_matrix: stores log probabilities
# viterbi_trace_matrix: stores indices for backtracking
viterbi_value_matrix = np.full((len(states_list), len(query_sequence) + 1), -np.inf)
viterbi_trace_matrix = np.full((len(states_list), len(query_sequence) + 1), -1, dtype=int)

# Set Start State
viterbi_value_matrix[states["s"], 0] = 0

# Initialize first nucleotide (Step from Start 's' to first possible states)
first_nuc = query_sequence[0]
for state_name in ["E", "5"]:
    if state_name in state_transition_prob["s"] and state_transition_prob["s"][state_name] > 0:
        viterbi_value_matrix[states[state_name], 1] = (
            math.log(state_transition_prob["s"][state_name]) +
            math.log(state_emission_prob[state_name][first_nuc])
        )
        viterbi_trace_matrix[states[state_name], 1] = states["s"]

# For example, the first column of viterbi_trace_matrix will be 
# [0] indicating start state released `C`: even though not possible - but we just initiate
# [1] indicating Exon state released `C`:
# [2] indicating 5'ss state released `C`: even though not possible - but we just initiate
# [3] indicating Intron state released `C`: 
# [4] indicating end state released `C`: even though not possible - but we just initiate

### Implementation of Viterbi Algorithm
Write a function `calculate_prob_for_a_node()` that populate a single cell in the matrix. The function will return two values:
1. the maximum value, for example, look at the 2nd row, 2nd column in the matrix: `max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)`. If the probability for `s-E-C-E-T` is highest (lets say X), then the function should return `X`

**AND** 

2. The index of that maximum value described in the first point: so index of X is `1` (recall that Python works on the 0-based index system)

- Populate `viterbi_value_matrix` with `X` for row 2 and col 2

- Populate `viterbi_trace_matrix` with `1` for row 2 and col 2

## Understanding the Viterbi Algorithm

The Viterbi algorithm is used in Hidden Markov Models to find the most probable hidden state sequence for an observed sequence. In this project, the observed sequence is DNA, and the hidden states represent biological regions such as exon, splice site, intron, and end states.

The algorithm uses:
- Transition probabilities
- Emission probabilities

It applies dynamic programming to calculate the best possible hidden path step by step. At every position, it stores the best probability and traceback information. Finally, traceback is used to recover the most probable hidden state sequence.

In [11]:
def calculate_prob_for_a_node(viterbi_value_matrix, row, col, query_sequence):
    nucleotide = query_sequence[col - 1]
    max_value = -np.inf
    max_state = -1
    current_state_name = states_list[row]

    for prev_row in range(len(states_list)):
        prev_state_name = states_list[prev_row]
        
        # Check if transition and emission are possible
        if prev_state_name not in state_transition_prob: continue
        if current_state_name not in state_transition_prob[prev_state_name]: continue
        if current_state_name not in state_emission_prob: continue
        
        prev_val = viterbi_value_matrix[prev_row, col - 1]
        if prev_val == -np.inf: continue

        trans_prob = state_transition_prob[prev_state_name][current_state_name]
        emit_prob = state_emission_prob[current_state_name][nucleotide]

        if trans_prob > 0 and emit_prob > 0:
            value = prev_val + math.log(trans_prob) + math.log(emit_prob)
            if value > max_value:
                max_value = value
                max_state = prev_row

    return max_value, max_state

In [12]:
# Write for loops to iterate over the whole Viterbi Value matrix. 
# Each time, call the function 
for col in range(2, len(query_sequence) + 1):
    for row in range(1, len(states_list) - 1): # Exclude 's' and 'e' during sequence processing
        max_val, max_idx = calculate_prob_for_a_node(viterbi_value_matrix, row, col, query_sequence)
        viterbi_value_matrix[row, col] = max_val
        viterbi_trace_matrix[row, col] = max_idx

# Handle the Final 'End' State (e)
final_col = len(query_sequence)
best_final_prob = -np.inf
best_last_state = -1

for row in range(1, len(states_list) - 1):
    state_name = states_list[row]
    if "e" in state_transition_prob.get(state_name, {}):
        trans_to_end = state_transition_prob[state_name]["e"]
        current_prob = viterbi_value_matrix[row, final_col]
        
        if current_prob != -np.inf:
            total_prob = current_prob + math.log(trans_to_end)
            if total_prob > best_final_prob:
                best_final_prob = total_prob
                best_last_state = row

viterbi_value_matrix[states["e"], final_col] = best_final_prob
viterbi_trace_matrix[states["e"], final_col] = best_last_state

In [13]:
# Write a function to trace the state path that gave the maximum probability. 
# This will be the final result. 

best_path = []
curr_state_idx = best_last_state

for col in range(len(query_sequence), 0, -1):
    best_path.append(states_list[curr_state_idx])
    curr_state_idx = viterbi_trace_matrix[curr_state_idx, col]
    if curr_state_idx == states["s"]:
        break

best_path.reverse()
print("Sequence:  ", query_sequence)
print("Best Path: ", "".join(best_path))

# HINT: You should first find the maximum value in the last column of `viterbi_value_matrix`,
# because that is the one with the largest probability. 
# The index of that value is the state of the last nucleotide.  

Sequence:   CTTCATGTGAAAGCAGACGTAAGTCA
Best Path:  EEEEEEEEEEEEEEEEEEEEEE5III
